# ⚡ Google Drive GPU Refiner - Remote Worker Node (Helper)

This lightweight helper notebook provides external GPU compute (NVIDIA Tesla T4 / V100 / A100 / L4) to accelerate video refining for the **Main Google Drive Downloader & GPU Refiner**.

---

### 🌟 How it Works:
1. **Shared Drive Integration**: Mounts the shared `downloader` Google Drive folder.
2. **Zero Configuration**: Automatically discovers the Main coordinator via `data/cluster/master.json` on Google Drive.
3. **Hardware NVENC Acceleration**: Receives 3–5 minute video chunks, transcodes them at maximum GPU speed using NVENC hardware acceleration, and saves the finished segments directly back into Google Drive.
4. **Fault Tolerant**: If this Colab runtime stops or is preempted, the Main coordinator automatically reclaims the chunk and reassigns it to another worker without losing any completed work.


In [ ]:
#@title 🛠️ Step 1: Environment Setup & Google Drive Mount
#@markdown Mounts Google Drive, verifies NVIDIA GPU hardware acceleration, and prepares workspace.

import os
import sys
import shutil
import subprocess
import json
import base64
import zlib

print("=" * 72)
print("🚀 [1/3] Checking Hardware Acceleration (NVIDIA GPU)...")
print("=" * 72)
has_gpu = False
try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        text=True
    ).strip()
    print(f"✅ Active GPU Detected: {gpu_info}")
    has_gpu = True
except Exception:
    print("⚠️ No NVIDIA GPU detected! Helper will run in CPU fallback mode (slower).")
    print("👉 Recommendation: Click Runtime > Change runtime type > Select T4 GPU.")

print("\n" + "=" * 72)
print("📁 [2/3] Mounting Google Drive...")
print("=" * 72)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully at /content/drive")
except Exception as e:
    print(f"⚠️ Drive mount notice: {e}")

print("\n" + "=" * 72)
print("📦 [3/3] Deploying Helper Modules...")
print("=" * 72)
workspace_dir = "/content/colab-gpu-helper"
os.makedirs(workspace_dir, exist_ok=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)

# Decompress and write modular files
bundle_b64 = "eJztfYt228iR6K/0yHcicoaiJD8mM8oqu7JE28ro4RVle3NMXRyQbEoYgwCDh2RGq3+/VdXdQL8AUvIkufeeRU7GItDP6up6d/X9xiRNZtF1f7Hc2GMb0XyRZgVL81Ei/8x49WdZRtPqRxHN6y/5TVlEcf3tJuPhNEquR8ksS+esWC7gB5NfzxdFlCZh3GMnUV702FE0gf8eJEtZehEWN3E0VsXfw89RMkqesSNe8EnBBsltlKXJnCfFHnubptcxZ4dpHI7Zbc5O0kkIAzkeBofnJwev2T7MpY8t9vlX6C3vjDa2YcYFVB5tdFmYTFmSFm6hw70RPhtd7HrKZyyY3PDJlyDKg2l6l8RpOOVZMI2yzgTaiKZhwfHXHsuLrMu2/szGaRrvjRIGz2gD//eRZ9Es4jm7i4obdpRFt5y9+TAcsEkITbMouQ1jbAdgw+5ueHHDMxYyaBPmnGZLFuXwk8qwegTsLs2+5ItwwvuiF9FjkS1l3/hEM5qjMVCWZsbEo9yZS1drAp+MF2WWsDdhnPP6yzP2uswLMZN6sDGAkdZfFTNHhA+sQIbg2GefeT9O73jW6bIZDAthgePCNtxBXdWt8K8TvgB8oH8AbGuNVx/5bp8d4qpSt9CDDu55mH3hWV4Xx+/BQkOo39IoMQfXg6XGYoQ2GvTVW5yYmjZingl80YENdRdw1WDWhqBs+cpagHbw+bqxWqCp/a3kJe9Px2J6Rg0YymhjzosQ3/Z/y9PELeTpVS7cZVZy8+MkBlSDbeesgpgggl8WMVagGmr10RmpuxpVX/aCNC+KPsT1sbvq58ptcY0laujU0xiBYB5iSW0trLoNPRhLom+h5/oWQjoP5JctsvQ3pNRyD7EOUBoOBPELoUQW3m1nfBYlfFrtN+jZ3DJhsuzwPk+mOZJLIMj9aLFMxkizKyjKIQsqjhWm+gajPY0vPo82oMvRBqKH7Ff8EJvyqoHO2XN1yUnL6piFBQfJeJ7Gt9zmH8QtgGsYzEL8fbRMwjlCNAaKCoytgGlJhif4h8YIKsrVY3m5QM6JLDe/gT8mZZH3RIt99WKrCLNrXuRb4+UWcBQaW4mT6BE085swg+WZpfEUF/DuJoI+r8swC4F1cmx5Bm8kdYSi7GzwcXDB7rKoANYqBwOjLlJzwFmaFn13opJDKa6tAXIc5oJf1Ts+HOf4b0f9hq8AJt4JAhxTEHS7zjoYxEI1SShQr0VehNfIsgTLF9gNDOJ9FgEaLxmx/z12WGYZYBjxXQRDzTA6MIl5WgJ8pqxMcEn0mcsxTe6m/pnAWsC3jho7tAU/+zCmrFA7QMkt21NscFuJL9gkAg8x3W7YrgR1ek7vTsuny6P1Cw8JU+jvHGpcmXJHs9R0N23Yd/ClXoCKvDQhbsdAL1WIyUIMZCkSDSVY5esA0RDWwZlMQy8amlqCot6gPh+XQRB9jKYW9W+qX7UTYG82u9Nr9bDVrpfau/KdaM7XESEnwCJKbKa7aiXb25SLKgqtLbotwjyvkeBFnw15mE1umB9RhUQ95XyhC9TjUhNB50uq4VvzCtub1ljWbV/emvVqq6tqwhyOlXjPbVGZRms2BlMO++xNlIFgjaDYA+qfChZLY0LCUwn+cvYovToyKz5UC0a31JijZ7FmZRy7gpWcQU+04EczE8VEO4I0NeKMLNSOM6KQzoIlbMaID7CIUxc44YT0nDQhOUSwL4bcgYGknvOtKMl5kkdFTZEfCaYkzeYBNbgP4+zn5biTbX4ejfLRaCu4+nGzxzbZpgQXkO8sWgBNVwJgwy6tWkTsnJAmOzXY+2ijAVJPXLS2hWvoybMu6vWj9vLLai8LxiEIt0TZXOMlvq1q8pqm/aq38oRNa1T/HbeS3u6/dD/Zn//vQul/EV4eVkQDRPWwjAsyQ+E66wKGFOJUWSmYNrGUbf/ERSNK2tHbwuEcnX86Ozk/OBpcBEfHF7QkfrVhlACoLIn55Pzw4CT4dH7x6/D9weHAHFvB5wvsnYNOIsu/PhgOZDePla7drgxsVy2TTq7sbfhDjKIrLHrDIisnAAegA4Ycd1SrhaPk4uCTNUbqwQQUKXeo5UHLF4M3x2eDozUrKY1Qq3h6/vF4MPTV15rGyvP0NiKxt657+XGNesVtkPPMrnp++a5a9bbaKdoGqebRweXBetOszFKng8sDqvaX4fmZU0+213PtNnLFBosbPucZ7BOytLKPp+ZiCbRoWDILZ+SaBZFoXlalua5bGxdPKW2XFwfDd8Hw8uDt8dnbNRsosjC/MVQ/UagJsJ4mKsiKb63wNRv3Q/lycPpedm31But9cfxRwcZcY2khj5A+j0vUQd++/8AOhXGH/QG0qDL5ksOGo5maa9YCMheRdFgdvvtw9uuwDeTaJ6IENAoNWrKF9SCt1T48+TC89G8WHbq6NVBVwSYHF94+tVaxNorYKE7rtU8P6J83xycOybNqG5a2esLNA3eRQx++Wb1lEk4/9lTMAi0T8rZkT8tGO4Vyh+RWKjNyaEhUCY4+XBxcHp+fBcPBIfT2/OVOf6diyCAcsjnon2jlWkALX/hylqEQkfNrdPawTgp8fA5EZxzGYQKaTzpjglcXaQw0CV/9AbSBO927crx9rjA1OBkgV7o8Ph2cf7iUY3ixI8fwjL2quo85KAtsKkfPOouM38IIcugOmCcLxyCNCQ9UjqVwR73cmqM1FVEUJM0wySfplARIsVTBu8HBxeXrwcGl1f/uK+j+GTsF1Y2JVYJpzWLgSSTyTXnMQWckoYrdgORcjHlYkCS6+wr27unBfwUHh5dIFtRC/eX89RAaflnDVdoRlYKflgDgaMGxkz0Gc9sal7MZ2vnKBVrsXrLf0jEa/XhlHmZ5nOJc+RwHheMEsvF+cPjh5ED0TfCF3X5yIuf1UsxLKGo5LtXO92gYvgZ5JmdARWJYRw7zgHlKsEU5gzFdR2OQA3CC+YJPyjhEZQ2kMeA5MY9BqoOXAqcQ9T7ybJzCkpyk10TY/sCO+Li8ZmdpEc1ArsKSObuErzFIb0eD1x/eBifnb4d75KCDcaKgw4VHEY1wndHG4Ozg9QlQ2aqsEFx20eD2HUpUO4T4w78CtE+xAG0e8vy10aJ8CYCcg9x8Xe8bIPxojSQfZ75nykDKmVmZGifpYqlYwH9+GHyAIb5u67B2zFSbvqmaS3qMyjjWY2lJ67EwvguXgCsw7pgGOBweseF/nkSApygjIgrBOsJ2QGuHMSfakVhmC2rSpGZhFIP8l/+JkVRzF0Gr2LI5TZ+sq83FnJwh4Wql7CaFVb5cCO+dKV0n/M50556lCTcs9LptfgEwhN2BKC3tkMBaw2tOOkSO8AgZtAglpxHQhVuOAq/Pdm84ca/jFAgds1mwlK3gD10qdEXX+p0QSevflZxp9lOtviHC9FgldPSYwc5d7g/vamZhNO7hW3VhjQVBF+a26lnL1mP6Alb6ZsHkivn92vJj1/HQqEYcrctWh1QLovijVRKq9FS1RK/8FNVEr/8E9USv/ngVhaD7eDUFq32DqkK9tgjL+P3JYi9W/hbRl+o/XoDVqz1NiNVbeLogS7Azd+kj+F69Mo/mYdpu93gKWyk9FtAcwlWlJxPXuomVVLankdbHENCeb7VFv1etZk2A5jz8woFc5Z1pT7gMgvTLPnq0NYPdmvaxnBcWdwQIe7mmtBAkOTB0oMcTEPTQc6r5+MVYRom9FN+4DP/cJeh5DWA9Zpg+qp+VOaPHHAOFKlQPztaKqzf6mBpVwVGiEMNEijUQogUZFCI8Q51uQv7vyRIE+3m4uAHxPQeZ+5NQXE5A6CukUqKWVI0PVS3x5eT8UwC6A66qW2YH9ZUdQI1pJOIdPpydHJ8eXw6OUHuZSO876HhboB1oUpTsVYK7bq9BPyLpT1YFXienAuWr0MV+NcGObzoANhhEQIMIvC2hxIiTIcmRlQlKxKggTzQgkpaDkwF9FuTmLJp8ibk+JygM0mIOC7LWKM3pS8n9MprztCxInRwS0D6FxeRmml6zi5KCOKrJAW5e2ArqL6Afa0Xq1bt8dzEYvjs/OQpev0dQ7+7ssB/gv89f9mkV8fevr7dzf2VTCX5OnRiAFmO5GFxewGZHYwEjN2pURCBP/gg/QYKT3psXIGKDQsnCAq3LRa40lq05n6OL9TLMv7DLLJQqB8Yg8TDewhBWFqO8Pk+h3TQjd3EBhVUIFf0doK5iAB30si9ogMfPwYRiPpAo7oh+3wqRVyrzqHNnaZxr+Ca+BOGkEC5pEWkkJL61PjYOSKg0eYhQ+jsniz0Z7/E/tS5jBxsNZfmcqQoEo0mW5vnWAhRw+DWHicyBR0fjKI6KJSGTGXoRznhhKjBKMsfOnZCc0cZtNOWpKuvxP41G2z/8+95o49/+/N9Xmz0aa4/Kdf1VRqP8RyrHqoLKXWW4XPDL5z1AuSscIUyYwNNlf0Y0ZKg7UhEFzfESCWAgLMXCHNMhVwhKLAFsTPLvtAT+SlcLNWMHcF1iq+x1pOjtaYquTxCehSZN8dBGBVA0U/oeK/M7ryzydUjADEMHJJ8q0Iyz4NkcSEyCMVkZbArsJSpEI/Gyzy5vopxVVi59bH8rYWshFuflnDhDc/iWCxVbG5MeWregq5npgXbN3MzDUpvEnTKJQJpUjs6Zsv3fY1R7H//zstPt3/Cvn/d+vnoI7tWQ0T5De8gz6AcFAHwIloAgeWELtJ5BaoPRhviM1h/tf+aip7joYsEp7I4k6rqaiL7v46p6RtnTRmb09QFYUvJF4JhofBrlX4zwLWvdtIZckVOgVcfbmRPRa4sbLATZUJc5gBYXHVikz4SFYvNcAePK0N2zB5JIGYuwN7E9GS0mka17FwQPe+yePxgxwa7U7E63HU3dqXvKu8W9wc1rieJWC0Zs6STmYVIuRK8UFxYAPQqXxAJA6Kte71XHLz5DgSspp3Q9BGs4CZPcjd5kHb+zW8TlUqcMFDbc+aQyGoGikwnQ/KQgo5mKFUVLAYdlrIP36mBUFuV5KQLJyXR4fZPC9pqkC+D7krxRtCrwHo2wQYtzVubI6z3E20+/agihGZpkN3U2xKNrKuMR/l2HlmltUOT7ykAzD2n0BJuZlioTcVvDzlZK9mbLz9AMDZJMkmtLOVUmaoSGsrPoayrQbKFq7uvaNbBmlBqgcCcbbfxvocz/MBoBfvR/+F/IpaHE8duz84vB4cFw0O01Vq0sPFgb0aqt/lU9I7SwI1/LeXYrfDnS5yA3zJ4qeDyrrVTTFCCKa3MTwroqEwSuqWFowt7zScmJHY+R7tc8G5HAt15IFwNo6zEmj2Zkka05gbS4/d1OqrCwhj5kP1YfsrGucUQCo09BYKw//pnt+EljxanrKTYxaPVIdobujhmtv+ikp4DnqVKzi0OxrlfsgtZmKk9MVYtIq6NRM2Cs97LdBy30u1orXPHHmx5XLBlW8i6arzdt2Zo7a1k76qx59cTnf/j6YTe9GqRPWUNj9s0LicVqXr++oFH1KoUMEfFfK4y01Ynw1dKEjAfDmnwqNEB8sSJOUASqeeK6GwM3nWBuAmzVjo1LbiA3jghK4nhMgu0KQfAFVr2Y3HTEKDx44V0qdDjgiRNiG7QkYVZEM1BkEV7Y1IODsvh49atqZm55Be0f99mupzVQib9UDFaVNdHbM/xDWbBcsHtZ6UFxwPF2Lcd4NDJxpEW5SS/KRExeMhg8gTBPpyUUl+dZYVhZmRCGRdKn6nNvrpDooDtgfxtxen3Ns8bDu6Ap+A/sovXWeNF+ZNc6qNurhEhpYaLKWyiGgSZxXbk//6CsH6MkgPeu3eJCGS5g4cfldQCa1jjm08o5jxQGufNAvEcpUYVn0h4Lc3wBNdE308NgSmDEQArQ2z9FPQpazvgEsA/7h42LE/mM80Dpl+ZydSVtPYGwTb4N3h0PL88v/ipiQ+TIK9HeLz9Xlhe0leN4OsZMTPFaTAbFCdC36M9bGcRAVRGCFMoQkgGGvMoCg3Kfe9iEnPhEJwIqiGuYbxaGwePo1Fi1vVbHRHz2xkNcUWTpLp4gnNmNkgGFgiRatS2BWOLwuUO98EB63R905pslPmv6MHBtMGqalsZvpbkgWTgXKAez8q8JhqiIMRhrIVUye5Q1Uigc6nhwyMQO9Qn0mRxPMIVou8kBbdHuTQImcQ5Ef0Rp1uH96z5SGeO0mRdTDERehSlGYViBhTyGT2fh0SAYFgFSkLwI5wv7EKOEh6Iy/eqPJL3rkFVuhj87o43v3+19f7r3/dA4aQ+kLwvQ5JRC/xil1DFh9Cla0JndJcAmQULCRiM8XTVBg6sgXyCSA3/DGJSYUTAV0VMAqtxV50OmLJt5y9H5AGlagKoqDhZ6AZT/UZgY2Q/sl1f0g16v70ehOWrtFvxrIcyHllWWlFoqQzEkoNylJRDgsCxSgL6MPck4zGOCMCgTnkzSKVGUD0mE8Wcgw4QZnYKL+XU4WTIJ1LYpA8/oi6769fi63u+zuMxvOubMO7LrQUL/zbIU6Oz5kP5YdRpjAngGwimIJlmn7gYlX5oZ+d97wmAhNOyymG39rBMaGiCCFgeNrAb+6VNlDmRuApIVjiPfxwALhBsecuxPufpuiRsOKKqmmwsaMHkkiYr5LY+DfDkfA1WmHz6jfXyLrJE+98vFoj6FgdZs/Ia0+fjszblxxqIyu4/K57svfhmVM74zU6DjsV6XKG9T5enPL6bw3+nO1KkM4gxskk8HF2ciYg7/Oj6DljzWXRrHT+FO2zgGFxfnF42T+OPLib/a8MPh4WA4bK6488qi2/R65/lzfC33J59HRLW1ZeixIryWf83Rt33N5S+gIhMglChWVqILmed8drVDND8ofoLdYHCF+HSwQHqWS90BtnxUubBEhGZPIwcoaBE/iIjFiHPbNAb6LHe6x9rVSPOVGb+eDbVJEDAwozKNWWKbpdEa1iV8CrQPeZiHac9aQqF7s6nRRlUalpVVx9nr7zRI/EZ/OJ9h5ahieO18kkuJn+Wfbm2UlLE6sjBiXNqQHzRuocma/ZAWs+NR5dDlpBdF9duWPi1YGi0v0kVnxzBVwLwCEX6KWs09/Hy4EsIZ/KkkMp1OijUoSAYUVfKHK3bvEqDuA7uXrcNfEkAPYqOoxp6R+pQBm7lOFTMmnittbYRK+CHhd4S25GbSwOZj+H4eXI9bMN7EstmgOGrK6yYc/Yk7dDuHc+7JaK672vxB2ytd4BobVZEihkgYFS/br/hXF60Ss6Yznv22qesl18gWonObBEOklx0/UTMp17vo+mYLljHN0PebaLHVbI7qaIH28o4MDKbl5phT4jrMpjGGe6czKUcL1cwUT2taq9gWUdpqPAZ9VVCXonU5wSijteYwFGXXHH3zECv2svYohdqxzhiPQPiJYsqLEl4naQ5CHm2YjoJpiiJspKBZayPNw5Uku22wglWp0d6FWbLWYJWdjHazCda1gamEhTUhSdLbWoMjefObhialj7XHBoKrTqU7FO+zhw57ERtDUpzP/uBVQ0VLglqqHEV4bKI6LzmFRkwxvpGvSykH3UMG3/m8JcZ41TQHEnX9E7FjSNS4ayo1L+MiIjaAQRiAJ+nMM6v1ZlC7ubzjd0QY/9nsSto7E9JTYpwRgYbTbAoDX3K059VVcQ52TiU7B5HNZkx+zD9vwpvNK8WU1W8fZ656VAJE1YKSgLAdi1XDZ/q9eWXya3gvERcqGfyilnyRkQjzM3Xb1V3KmUBlc2sRs8812RRheRPlzgGGZuOCLs1QP8qeecNjUGhknNPjzZrkF/GmK7QyFJZZHEfjfsb/VnIMSDVfE5mxXy5g0lrGw3K8yFJkKGuaUD3GU5EAcVSPWFnD5E9h3NX35vWiDKJklgp7i0lHjDU6TvDwFMjbIlaE3QAnvsM8TYi4Zx+Pj44P6IAoljL3IDZvCuCjDeyWQqJgOWD9oZ4I+ATqqBe7zcJ5MB9jqR3jA9mwM1Fb9fPgsz1gmOK+Btu+SHgArxdl0TE3yefRRnIbAaPcyueRUDi3tmA5s+UWjHcfx9sTCNqn6MCeGEUg4ylVDUGw9if5bS9JbzjGMsMfZYKBrBtXliqA9HDfDN2wYszwWWDSJiQYC/VN+F8o909Z9PNFHOFBsx7mS3I0A6oNKsE+e2HbTGFpPuurcUV2OSj+ecfOzCeKViuCJdHpIUrvXtmuQ1FcrVPd7vOrNc1aGk3BxjSjawI8eBKEtyDYoKjSYH6lLE85wuDs4+DskN3w28n2zfOfXgpxGQ8nIvrK1GYkCcGGAwFIkopme1Y7TsGsZ7P5gl9LfFC94drXy91D4xtQhX2tmaPBx7MPJyc+moqDF/MWOfZwCGSvwglZH1bC1w73mcQY7/SOCKXYhh4jw0EJQm06x5OeuM0vKHw0U3HaCZoGaydUHk3xEKbIoSooMFMZ+6T94EieW8vZaQjjvo1ClSLuSHgsYVTRXB0gfa7HalOYpzjhqdR2AeScmqkoE627sGJkUpjACBD0B07VEeDcyCynj8Bj50CN+vD87PDDxcXg7FIG1WMgsVIYydAD9BREvqCT83jWk9gURFO/nwfmmYKQgCcb0ywAxtBUTHjtRP6hQETftsVd4YMD6CsCLwyhGr3XymGYFH5BrNbrmKShLw2cQGcEvRcIjvbO+sslz+Ow/qCoFeHqGSyRLphQXxV8oPP6byg/UyeNgvtqgCpHzENDkKXTOuKlFqApcJx17s1pbqpZgkzTddoQp5X8a2MtnppvPxN/jDYoqR76jc2CIs+ev7KQ4YQT0BiI3F+GL9MqIoK8AzpbvSfYORDpWkCwNQWcw/3OnsSxXfHHQ0ujzZHiRpUZKZ2B2KMBTDTGWHU5IhJZZnEaFtT9g2n5uRDZaYCAAVObRIXMdiMzb2E4QZYiBZgafM7ZHuKsu/1WwdhOpadwpLGCWb7K0WuFp5nt9dwPRnBGczxfT7bQfXQTZkKrqh2zGYu1U2g+ZW6tpuUx9DhplCZNGZSac9kS7IQz1n9K2vaX6I8WiGE/a2a1xceMfH3GyBAF2JyW1zfAvtnFwekWRi0zeT5zjw6Pb0/57XZ+M6dEskrwIBS7ASUaJKtXOzvs9DWbZRxECoyY18nrXLk8YblkQw6VoejSAKUr6n3fzjLty0om2u16cg1X39ayVsJUSaIRYVbYf1Ciilc340UHrNanGUsAiLM68p+GNaDJipBuPFpjZWKVec3mlX1TU9+8sT7N8HOTPeOjHeR/wsiUK1HgMI7PTqhDnM9KkNVdNZE10PdfMzgPtP41A9Ht6Z4RuDb0Wh5TCRKCmo9TE6Q1GPxcm6ku9eEz+HoTYvbPW05RFJhuUCaEJulV4+SbOdOON7MxDycYI0gxAloHu31FCH3Hp4V0QJ8bc+vUbWFK2+Ykwv1tDHHclue+t7XBCcIBahmlwum3FKz7etGXUr15VKscq8TODcH82z+s0zrmclT5pZ2WGlLprtfyq75KDymTQdqt64xzvSbfoN021/OQOLcVmHcwgIIhD0SJFK3T7XmU55gUmwIBVWbhfjMa1ncDLER+F7L+Sln085UpSOFdBIRDMkmxyB1iWhxxn8qM5j4U26g3r/fzlUUcFlq4hdr9+EOGWFgmAhDlPJTOmqMyWi665uwA6UUym4xXCX2M6TW005SVUIpTIl2EkSjBzVig5wN/Wjf9vp4wfo3u9KnDHvw1AbnJSl4sErqT5blOMimjQTWYu9RZxYli30aQgXrqiNEGebdFkl2jsCWzrlFjRVZt9Xjk3Kk4kJPoc/ILu75Q+6ak2oQIjbmum+Vh72Eb0Yu8BiXXElD721gzhax6gFR7brsQeb/bsvtKqFhiJjbWlkiVANO+Q4TIhw2tvyE8ct16GoCeohUfvAJoa1qipomppeS5HvItZVO0lFdg4zwh80ih64Ly0KWpDlpOncnCUK0k5TaHh+ltPSmCJguXZFJRmZ0fB+WVFXnSD6fTDpb1wMoZtVoSWd4cv2Qe0J1Tz+obdfdJmeVCShp7LvERedSr80yUlwDZor68JkNRhm9rK0M3GrxUZIUYqjVjcdMEViCpCv79zq0Jr73HErB7BR0s42FX3pG4ZSt7PUoAGXB9aZZfR9Nu3GLtmrZGU2xEUs+jFWcHH73XFzzuhps6vkXUpmxVT4hrwYcOE+6T266PdoXOrGHmaITbp+IySV+tHjQqmjBdZbsDmasDf3fNCzVuimJhBifaDwUNB6Lzqg3DWthc9xk7r1LmRdcJBkLtPv9jfwf+t8u2xYlrOs4a5nRyFQiazNWE5AdpRgbayI2urajz+q00fx2JgYACM6gGpO4CUtMlC3A1ROvrShbSyFtr2MhoHDzoY2TFE+sqFbHv2HCZTG6yNIn+zqtzAZ5UXr5HHBuMfPt91W+5LXycSwN01YFu0DFOTbp3qNlPO0nQANpshKu6am/nEbRDPabxzX5UAnOFFd/I5a1m1akafIV2AdwRZTLhQZGKI1HSYYNWrTJXyTdHG9E05uaelsFKIm52ksZAZGEsW2QJkzlapNpP0bQeB5OuPPdtJU+SMCdgtfKOoNvb9J048aWV36MqW71xyuq++Bb3j1NPc87b1WovsVNLgJcq0V9u3C3s16DKS2sFyTqlM36NJ1kAqIFTuC6reTTUPT6OHuNSOV8+rI2eKTkrEuLLsGcmy7PH3mQle0SvLfmdrb7rri3pFKUDJMUSLI8Obi281je7xh0Frdpah6ir+/kMpH7wH0qm9iqJQbRMs32CxEBiwrScL+i+PxhLD/OEgRq0/7yB+M3c4xirAaYBbpYD/+lAK5gXKe00MYQnOzTWrKwLcxSaRREMwHaCDKNXMx4o8pi1k0Ur9AIfm55V15QWN2Ghh1hs5hURNimmYHloEPPSy6rl4xmmHLoLc5nUeirPro2XhoxDDEVY42QSrIQyaRvJIbYmIBwVIolRNJ8DdsOveCmjB7YUnZGXreJpMK2LFpPd/9Cb2viUB2jwxJVw3Fv/PFrUvEX1W9RQ/GHmLVttht7fSVRRz+9NL/3mLNFLk0BprJbfmYaPxyv7SPojB1j3Z5WXJ9Ll4YLRxrvByfsB4ejMOD72kh3Y1ARJg5oE/BT0A60UuNfx5WYbKDf7/b4DThER4cqORB73xT+u1cYOJPEZbTDCQ1KZAImMdFjZrMab7skO6cJ3bhYnARXh/XII9kWV60gn0DpZdjMVEIUuC9QfHDH2fwjfNxO3fwAZeBQJWCuQw5dtQzb7zbJNnQnN2+ITBJ0ATTSBjNKWog1PppRvXx4BQVFwrzFiSsXZGd5j//kLfORJITqEGSbs3eXle/b+fHiJ9pC3g0smRyKkinGW3uVAvUTQsEjzf5tGeAVKWk5nMQY1jjFDnbzrwNlzks63UxtDH1YvhUUKkOreqvxwr8BjRNe5qAEzCaTqajaNj5qRo9YKGH2AWW8dXNOVXhjSfZr+PYrjcBtv/Oh8ijDRac7OLtnuTn/nTwxe/PTyT+zrTy+7eMA15p/4+Neo2H714o/9Fz+xzq/vLk9PesCav3D2FuTatMsOb4B28W3KCov/Y8NwFmaRrNLgdDqYIF6JEYUL4SOA9d4W+8tb5b+2REDs1vGRq6WbFR4cJkHQi0SiMO12iAYgV7pLTspLVx1KrzUft7pcBvT2Cn/a1uVywVX2DXeOzgpjom7joEP/Qm4leC22zr4aY091ty//tQ3ziPJWa/CT1Dr43SNVPi2L/Rc7XWHSzH3+4ojuwcbAeGS9eKr4+Y4v9RQ+43S6pASvUB7jFzvVUf02qIm506ap7Mp5B9tyD+Xr5z36uNvFsTErR5QcN7KRMJoHmPV3tKG8OhU5ckciRSE6UGcJQkRZ7jHH3ZQ/IGOu9y1mS8LpwtjdhEleauCbzIeLk6fPhdQocex5SjHqVQk0gvPq8sfHzRmQOAE5vErr+DtNW6a3ptmuQztXZQSrIPQN0xcnPq351z06E19zxg22Ua8cWgVNefX9i1pBFibQyuxJ+7wK8ic8PXh/TBghpMjXJGM6zAwGESzCJe62/2/MoV6T5HoKzoTvMgVkuq65Aqtu6+g4/LtrazEZuampmCkNjTa2w0VURT0pJBBZMau1MDMPZJQ2B00kufSjKQNvVyR4qA20zgWtIJpF1yLNXV1dvZRw7lnLZSG1BJ06tW3BznAJ1QNpAB3s2s171f3D5ndWIgB8nhnuIzMETmYoq00DpC55wgOR+6RJbklIcyhoxK/rQNHeO/uaKk7VzbF1HesmDUePhYngRQNUyxXYnhYoJGvac2ngxnVAUUsg/IpooSt/00aoT1OQjzZi09/eGuejnvUcbUZ0jhPks0YDcoDGXcmKXeAOs2Ftl1mzC3wwsOOJkUGeEVuHBaDtx4wFHwNFsYHHVW85L6Ce392FiaaeatQrWmsYn9Lh6slTomRtl7vh/trXJiAbwNTKu6aqtvE3I32rR7luswGH1mSBk12dACv5QbulQBwSqug7SEJ11/5koU+y03ri4pvE1NGGuBlXCD6CCclLQGWquT9JkJN4ZPIU4f0YK/nI4EfthsjKQePIesJwWMt6OYyi9rdKk8gcb0KZ5OuZQbynxvHRjR90K0p1WyednKnuHi1SWi1HAqylPzketJrcPzgFPhvS4JWSbjyK97qyj+Z/7qleunbvdC+D/GgJPz3mWQDqttXTpozIpozl70LKV3UeOz9CehWHekf98oK9U3PdE3c6s2f3eo+bdIgPoPh1s/sA8qXxbTEpNntsp7/T3evvzh6+77L/YGaJ2SLHElAV/vLhY8ZNT2SlRXpOaayys71P41geKEbGm2DKvvq4lrgTVkh4ePbXwbdKWyVLmIEQ9bD+vUKrfdvy+odKTN2/13M89PHKExFfokmyD6ay04CYakyrxO6bMJe6t80BfhOHa+uiv6Vj2ii4LGbRCS6zUbZafCy+tWuVf8bolD2lBavdBjIpC8jc4hio+Ma/hkAFqT2rU3k6NCiTIooVFPwnSGlUHZxRTwy2K7DP4WJaGAj7N6uLZu8L3dIskE/vw2eH8yjT2q5bqcu9YoeIUXi9tdxzCubafsMFYj+y3Yft+itlnpBgwQ2Zzthf0rFeXywvbdZNrV5UxBxa3ITS8G2z22QgyPjq2Cl1jL7epz0muqWsPj1WzUH+TstCjFmmPBZJDjGRGeaNB914TxwIxozn/Z1mK8OwHOOFcHIry2EgL0WAT/ganOSb7Ahq5+zJ2ToFtO2yVwPBKWaCA8uab5wKOqiwuPHbZ1NYl9EZK6kzO8c8sYKR8JfyhmxAaUe1TmdyxZ7dV0B50JevI3BYQPWhaxssCAkps3SbvUGcYwrJas+nggmHMdp5lyCQJmi16ZpWLm2/r0TiZqSU886NKQMmyvbxNnK51wUQxiGQTDwMJlMNS8u1uBSgsFIe/eux95vwS4fxU9Dr+W4oM8uyC9FUTTGrERKNrCmhxCIbzk9DKmnEEl1rIjgdLVX374mRUDpur0tROH0DcWi1ooDNGHWk7mtkWXgnscaMJfUal2oDK7TOQzqtLjUJ0DDQtVSD4umOQytyp9FzqMLaN7eB3/i9iKs9SsqFdG87Ccn8D3Ld2/cfRFaPbVsaG208aOvtaq5r+59evnqE/+m7Nv+TH4LGgLRzEBq+UJTBWEYzIq9oimi0rkpJx78hdi8EB7bPQsjheI9waJ3773rxFPuzuq4En3W8Im17n/BYoL7EZrXdRNJGUEXIvN3q7XACYoSTVGxYuVeRNO1ZSmwPL1EtFCtAqWSdSMfBVz4pMYZQpSDaQmYUc9D3gWyJNFSg/Cc5ZVLHHOp0gnU4xKtoF8uKTWqEOZc8AhA8Aj1vUmvNeU9cxCjuLgPykET5DXTTn3+5NWLO7ahJx88S3sn4IAHtfYLIZ7pq3pBSNFsrYpO3ji3ZXOmbfRZMZteyrCS0dA0pZm6Qmepg7e/1bStoed2+4mFauzUbqEppjMw4FC3eS/O6cx+jBQiDRyGSIy22U1X4chEA8kYJpcu5l1j0ENzXXZvIKiqLlGJrNY0QXtF23TrKFDNKo15NoVd3aWuKngsLH3eSS7u58Pc8o7nbZ294MbkhbigTd+2xy2xpmsvojlLWITJKgtS2OG+oH8IHfXEWxrESDaaabU5QGR12ZTKtLx71hqzK+4HJyxAm17zzcjVMbTTz36RlF/Lfp+WM0R+o6TE1PxP5C2Q2B+85TdibwDtuMJwaSJlKaSDzRVSFzWb9uGGnCVHHopp3XDP4Fm1Wbv3Io8fcvHbAmDElNCXkMeeLzm7fzJIOozNXYL10O/Z1ZiYUejW5WTF8H0vFpyVk4vMQqAaraMeV3DrEfqS1pgP8tPsn2iZ04aXcKtoe8cXF4tO0Yxw50yrov9O3hgIWsDeI9hVEdHtvrDDDPH8xe9GkRiitrBac+2xI064Tq6wUwH0QEv3whc+6iJ+2leKyrSnJdgzWFAj3F6V5+fQP1UszFsk1oJa+EXn0GzfYMC2RoDiKvgArdrnkBd7YyvM8ovuqEhVGnAmgSRVPxzpxzWIDvrUYCj93pI5biwdddAzodsEfmbgk3tusaQx02mo0CXpk+1kw16/O9GDwNutombP8MQ6NuPzHnVdNuNy5x77JNM9OX3fJMAsAF64ml/fVSG0HP1C+lddlFE/ZmzeY3FSTYyfpfG44Z3OYF10VLy6xl3Kf0qy1j8JEk5SxeeR6ks2cavAuAC6ONV7oZ7NuAxzDxClPOULFN9HLu/7zn14BoN8NPh6at2rmAWWFRT0f/hUnk1WzdGr5AGtob+u6aHSn5KuYDNeTmNY94oBduRJC1Y5nP94GoGjJMImtyd6tmI6e95XSzFLqikL8WLyULyd/E+Uxygwg2LWiJRqSofl7BDX5K8FHtJzN9IadIdzybDkL8aC31qenv6dNXsuG+8+c/KunTF7D6/lUtGvkB8ZLGYB9yF9LA2DPmHQef7w4OK0T21LMKlJJ3MZnH48Gh3t4YxHeTYbaJ6bTroqM06KIecIn5j3sTUCHMQL/LTDfBQLg5o40WJmRpZyGwobaUJqSZldkTS+IcQU6Vfhuv9r47QO4FZDW6jb2bzTzg1hHx8S5Fbqh2/gWw+yDhF9TKqmgyFXamS88+DvPUl8d5cyWaBct+N6uU67SterXVwZZmJZZkBMBIwdIR9ON5Sds//lLdIu4eaBOwjIBvUjSZJlIWhiSyOFe+9jnaRIVqRbusYqt/BTugtyDN64mOrVPZ03MBvUh0wDb2byvpiMdUOh5omjV5gy8rGOKGU4AI07TTMD9nqxmgA09eaednlb7/fH7QUu6bSMp97icIW/e3zVALY6kj+mCKVOG0FCxzLJgMZEeLOv9bJF73+cLTuGPmDZ+ZNyA5GoOdD8jXn4BjAinIK+H80lwWGqf/vGnr8UHc8OLAlr+EhjnfnP+khXHgeqJCkwWzcvM9Njs590rNZ6WUDJpwqFoAUqxLNv11/img9LUQOyFBK1MCyyM5WubaQMoxSSpAW2a4ve6Y0RLEKJhUD591fDEBOXx982hZZXEcIluQRPbbDfY2dnB//sEaqOS2CPAqzq//NLf6bF5+LWzg390qja36eUuvpQ0sNsl2VgSwab2JVShBw2m+Ot3Qjatxporh883oyg+SXrXRHn0B/Ckg0W3DJpFdz/g2amWrqGo5tOzgsI8h7b0Z6Vr0a2wlqPcrUZLu1ct7MrytLB71bKuLC+XdU9b1NV1qowmejRWc62HFgwmJSFKoMlkwjtiTXp0M5MwE4o3KlR+jGnxRURNa9IpfNYLTOHTXeZo+Q38nroXV2gbJxDuogRvhqiuWuj22ZuMc/J0Q5tNyrz+EH+Tt+0WjciunvUisanNuxDom/LrPXcCh+znKfHB2M2XKI5Xjfpxpnnfs9688VnTsN/0tJv17ecJQMOnPbAaH+/Jd/tZEU6AUu26CK4E/D12rwiOiHLEIEdFUvb6OzOKbWT/zYZIL1RpIh7+mONqrKZcCYS7hfeLSAQ1pn0g0I7upB6Nz+42c2SbW3qK1nvGPenYaoVu3AVyecS1e6vMi3UXIAYg/WhYKEV1/a7gf4CV8PEWQp8/Gh9xn8WkL76ThgW68U6rSRzJQ7NJnL42m8S9kBeXHzeBt1b+pIfgkfob+wzDx7vIphxWyJou3mD3/9KCqatp1YgoB4Y4Z6ZZQFsiWPGuXU/vVYpsvauXmMJ7sdSuLRLWdd29r7tA67ptF9BazvuWG2htT1WFYT07AsD0OsvIRI/R267WbPpuP9rXbvc2jNMEOTSFq1DTyhz+WDsEG4ZoPK9jKmwj+au+CIJcOp4hCzfMAFoHGxzw9iqYWj3+1Cd81ENJ0COdZiGeQMS88tqyPEri8EsXj5AkmqWGJyQIaji8Iu91pICIIE5ha8lsZVr0jid1owhTF3cYCT8+Q68TbCO6whCXvQMqKHtOf2PYXIZfum7c3DSGNYzworl9PfhJJB02bm1y7N6rA+5oGWrv8wtf+uEqM6q1emFeuQm1IxWeQHmMfvKsc8PEqgmiidF/O1RTZJpd+LNcpCsZt+NWW3XEqda2fM5BeTuzBIIR+IXdVUiyL/9tyjP4+81SD6P/tm7QfB8my0Dms9sH5XDZyfXkIYTWucgSbjXcvw3jkudeK5zES7P91rmucQKtsRu5Rk2WKw31vapay2U+Wt2d/iub/vjraTj/I4jD3lHr+wLKoBimvfoeRA+f7KWeNuXEkX6Pod099gmkbxR9iUpVR5n0KxCbFGqLctREEyhSQw4w4lryEA+nYMaIohTTBegq6GnAkRh3MSK9lBUEbtkUstnuQGZG4Ps//exy/VVmip931DU00L+81Q+vFxQeC1LCzNNWdtBdSy/wXTKI4yPVjh5V/Ih2YHR77SKG/VGmisBvp6/J7dd9VI8iyPSd8hQeqPhT5LLsfvOvPN+Unmj3FlW6d3DzLGUd9CC+kWFy3c3HzflQIcdkqebuvTfzQaIvcF682DtMeFrmINJKXvuoPo/o8vqT9FqeQZH3oOCMB2cHr08GR6zzkWfjNOddmr9sEMAg9qKa/NHxUBY/RdmpSBO6mCderg8FA611UY1Q09w9+s6pJl77gEg+8d0pQRJKFXPYDGNHpmP6zY2X9JcM5N4XZzlsearH0I2638EXvS5meAKRLvHl1iyEc8IWMeQk1CUKhR0T0S4BrSYHE/jvbBfJQW2CHKhznOL+I7zdXqGilhjFr3c2pWDUZDugyE+S+SiQVd17y+i0jwiwdMLIPlycPFFctEpUkWKeS9D83H91F49bl6M6Ez9NU1unddZkrbWxC7fIBdri+Xm8BIO+yJLPa6++Zz+1sXkNQp58ADL8UGfsDgLItJv+DMiom9QXkq2yp7dRqvr43stf2GW02MOUymFZpFsKYZYgmn/BrKAgTODpLeiXkh/7UBaVrZS+blpX225i1DCwPAz9gHrDgi/Y8+Zha5KLLbqpp0nvecbeOyclKPF+ES6ZlJbRkSKiRhgZhMTRO2B3TNJl4EJKtRC02ezjceK6LIMqB5HyQqQDbZfMSTPTxXn7NvaZ0a43Mhyj07GXFQXxeaTPT/f3hYV1qtx3Z21d0fD7VXW1t+3Vpf+vqki/xRHw1nrSD1jVo99r1Kv8gVVN+aYnYzZWVH+Ea9B2CzYQMf9SGT0JvctuTw9jF1HsUhcI5KWbmBSm49xGrvK945Q9d5Drn3mCgqSQqGA3XOfGTdVKFhEKwoByIgkujagqZXgh1NdXxWvKhLIH8kLKbE53csJSqNo3brS3J7Zv/fZMbt95IzswLWOiO2T5ihVJ89avfDlOQRA/ToDsZeVCD9VZiyijM1bCRSokkWpKuF3LHNpgw5uyIE5C5mWLG3iOPYrxtujt6WyGDieHPMua3pzSLf4h2zUkLXtAwgKCahAI11ZAGBgElVvLxk3oY+Ph/wC1KHF3"
try:
    raw_json = zlib.decompress(base64.b64decode(bundle_b64.encode('ascii'))).decode('utf-8')
    files_dict = json.loads(raw_json)
    for fname, content in files_dict.items():
        fpath = os.path.join(workspace_dir, fname)
        with open(fpath, "w", encoding="utf-8") as f:
            f.write(content)
    print(f"✅ Deployed {len(files_dict)} helper modules into {workspace_dir}")
except Exception as e:
    print(f"⚠️ Notice deploying bundle: {e}")

print("🎉 Setup complete! Helper workspace ready.")


In [ ]:
#@title 🚀 Step 2: Start Autonomous GPU Worker Loop
#@markdown Launches the helper node. It auto-discovers the Main coordinator and begins encoding video chunks.
coordinator_url = "" #@param {type:"string"}
drive_folder_name = "" #@param {type:"string"}
#@markdown *Optional*: If Google Drive auto-discovery takes time, paste the Cloudflare or Gradio link from the Main coordinator here (e.g. `https://xyz.trycloudflare.com`), or specify the Google Drive folder name (e.g. `colab drive download`). Leave blank for auto-discovery.
enable_debug_logs = True #@param {type:"boolean"}

import time
import logger
logger.set_debug(enable_debug_logs)
import helper_worker

print("=" * 72)
print("🟢 Launching Autonomous GPU Worker Node...")
print(f"   Coordinator URL: {coordinator_url if coordinator_url else 'Auto-Discover (via Google Drive master.json)'}")
if drive_folder_name:
    print(f"   Drive Folder:    {drive_folder_name}")
print(f"   Debug Log Notifications: {'ENABLED (Verbose)' if enable_debug_logs else 'DISABLED (Quiet)'}")
print("   Keep this cell running to provide GPU compute.")
print("=" * 72 + "\n")

try:
    helper_worker.run_helper_main(coordinator_url=coordinator_url, drive_folder_name=drive_folder_name, enable_debug_logs=enable_debug_logs)
except KeyboardInterrupt:
    print("\n🛑 Worker stopped by user.")
